# Evaluation: grade the pipeline against the generator's ground truth
Not production logic: a test harness. The generator injected faults and recorded them (`ops.fault_truth`,
`ops.supplier_truth`, `lost_sales.parquet`). Every check uses only data that has actually arrived (cutoff = latest arrival).

In [0]:
%run ./_common

In [0]:
NB = "90_validation_ground_truth"
cut = spark.sql(f"SELECT MAX(arrival_date) FROM {CAT}.silver.pos_events").first()[0]
inv_cut = spark.sql(f"SELECT MAX(snapshot_date) FROM {CAT}.silver.inventory").first()[0]
print("arrival cutoff:", cut, "| inventory cutoff:", inv_cut)
T = f"{CAT}.ops.fault_truth"

In [0]:
# 1) duplicates: Bronze holds exactly the injected ones, Silver has none
r = spark.sql(f"""
  SELECT (SELECT COUNT(*) - COUNT(DISTINCT event_id) FROM {CAT}.bronze.pos_events) AS bronze_dups,
         (SELECT COUNT(*) FROM {T} WHERE fault_type = 'duplicate_pos_event'
            AND regexp_extract(detail, 'resent_in=([0-9-]+)', 1) <= '{cut}')       AS truth_dups""").first()
dq_check("duplicates_bronze_equal_truth", r.bronze_dups, r.bronze_dups == r.truth_dups, f"{r.truth_dups}")

# 2) inventory DQ: recall and precision of the quarantine
r = spark.sql(f"""
  WITH truth AS (SELECT to_date(date) d, store_id, item_id FROM {T}
                 WHERE fault_type IN ('negative_inventory','impossible_inventory') AND to_date(date) <= '{inv_cut}'),
       flagged AS (SELECT snapshot_date d, store_id, item_id FROM {CAT}.silver.inventory_quarantine)
  SELECT (SELECT COUNT(*) FROM truth) AS truth_bad,
         (SELECT COUNT(*) FROM truth JOIN flagged USING (d, store_id, item_id)) AS caught,
         (SELECT COUNT(*) FROM flagged LEFT ANTI JOIN truth USING (d, store_id, item_id)) AS false_alarms""").first()
dq_check("inventory_recall", r.caught / max(r.truth_bad, 1), r.caught == r.truth_bad, "1.0")
dq_check("inventory_false_alarms", r.false_alarms, r.false_alarms == 0, "0")

# 3) completeness monitor: late_data store-days == injected late files that have arrived
r = spark.sql(f"""
  WITH truth AS (SELECT to_date(date) AS event_date, store_id FROM {T}
                 WHERE fault_type = 'late_file' AND regexp_extract(detail, 'arrives=([0-9-]+)', 1) <= '{cut}'),
       flagged AS (SELECT event_date, store_id FROM {CAT}.ops.pos_completeness WHERE status = 'late_data')
  SELECT (SELECT COUNT(*) FROM truth) AS truth_late,
         (SELECT COUNT(*) FROM flagged JOIN truth USING (event_date, store_id)) AS caught,
         (SELECT COUNT(*) FROM flagged LEFT ANTI JOIN truth USING (event_date, store_id)) AS false_alarms""").first()
dq_check("late_files_recall", r.caught, r.caught == r.truth_late, f"{r.truth_late}")
dq_check("late_files_false_alarms", r.false_alarms, r.false_alarms == 0, "0")

In [0]:
# 4) supplier scorecard separates the hidden tiers without ever seeing them
r = spark.sql(f"""
  SELECT MAX(CASE WHEN t.reliability_tier = 'unreliable' THEN s.otif_rate END) AS unreliable_hi,
         MIN(CASE WHEN t.reliability_tier = 'reliable'   THEN s.otif_rate END) AS reliable_lo
  FROM {CAT}.gold.supplier_scorecard s JOIN {CAT}.ops.supplier_truth t USING (supplier_id)""").first()
dq_check("scorecard_separates_tiers", r.reliable_lo - r.unreliable_hi, r.unreliable_hi < r.reliable_lo,
         "max unreliable OTIF < min reliable OTIF")

# 5) lost-sales estimator: regression guard (known lower bound, ~0.70 of truth)
warmup_end = spark.sql(f"SELECT date_add(MIN(snapshot_date), 28) FROM {CAT}.gold.fact_item_day").first()[0]
truth = (spark.read.parquet(f"/Volumes/{CAT}/ops/ground_truth/lost_sales.parquet")
           .withColumn("date", F.to_date("date"))
           .where((F.col("date") >= F.lit(warmup_end)) & (F.col("date") <= F.lit(inv_cut))))
truth.createOrReplaceTempView("lost_truth")
r = spark.sql(f"""
  WITH est AS (SELECT store_id, date_trunc('month', snapshot_date) m, SUM(est_lost_units) est
               FROM {CAT}.gold.fact_item_day WHERE snapshot_date >= DATE'{warmup_end}' GROUP BY ALL),
       tru AS (SELECT store_id, date_trunc('month', date) m, SUM(lost) actual FROM lost_truth GROUP BY ALL),
       j AS (SELECT COALESCE(est,0) est, COALESCE(actual,0) actual FROM est FULL JOIN tru USING (store_id, m))
  SELECT SUM(est)/SUM(actual) AS ratio, SUM(ABS(est-actual))/SUM(actual) AS wape FROM j""").first()
dq_check("lost_sales_ratio", round(r.ratio, 3), 0.60 <= r.ratio <= 1.20, "0.60-1.20 (known ~0.70, lower bound)")
dq_check("lost_sales_wape", round(r.wape, 3), r.wape <= 0.40, "<= 0.40 (known ~0.30)")
dq_finish(NB)